In [1]:
import sys
sys.path.insert(0, '..')
from src.search.engine import LegalSemanticSearchEngine
from src.loader import load_json

/home/gshjis/Python_projects/IA_NPA_auditor/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


['свой', 'помощь', 'родитель', 'ребёнок', 'юридический']


In [2]:
# Загружаем статьи
raw_D = load_json("/home/gshjis/Python_projects/IA_NPA_auditor/data/processed/const.json")
articles = [r["content"] for r in raw_D]

print(f"Загружено статей: {len(articles)}")


Загружено статей: 158


In [3]:
tagger = LegalSemanticSearchEngine(
    tags_filepath="/home/gshjis/Python_projects/IA_NPA_auditor/data/raw/base.json", 
    training_corpus=articles,
    tags_per_article=15
)

2026-03-15 22:11:01,450 - IA_NPA_auditor - INFO - ℹ️ ℹ️ Загружено 77 тегов из /home/gshjis/Python_projects/IA_NPA_auditor/data/raw/base.json
2026-03-15 22:11:01,451 - IA_NPA_auditor - INFO - ℹ️ ℹ️ Загрузка модели USER2-base...
2026-03-15 22:11:01,451 - IA_NPA_auditor - INFO - ℹ️ ℹ️ Попытка загрузки модели из локального кэша: USER2-base...


No sentence-transformers model found with name sentence-transformers/USER2-base. Creating a new one with mean pooling.


2026-03-15 22:11:01,453 - IA_NPA_auditor - INFO - ℹ️ ℹ️ Модель не найдена локально, загрузка из интернета...


No sentence-transformers model found with name sentence-transformers/USER2-base. Creating a new one with mean pooling.


OSError: sentence-transformers/USER2-base is not a local folder and is not a valid model identifier listed on 'https://huggingface.co/models'
If this is a private repository, make sure to pass a token having permission to this repo either by logging in with `hf auth login` or by passing `token=<your_token>`

In [ ]:
# ============================================================================
# ТЕСТ 1: Проверка тегов для проблемного запроса
# ============================================================================
art_44 = ""
for i in raw_D:
    if i["number"] == 44:
        art_44 = i["content"]
        break

tests = [t["content"] for t in raw_D[:5]]
tests.append("'Как наследуется квартира?'")
tests.append(art_44)
for query in tests:
    tags = tagger.get_tag_recommendations_formatted(query)
    print(tags)



In [ ]:
# ============================================================================
# ТЕСТ 2: Поиск статей по запросу
# ============================================================================

print("\n" + "=" * 80)
print("🔍 ПОИСК СТАТЕЙ ПО ЗАПРОСУ: 'Как наследуется квартира?'".center(80))
print("=" * 80)

query = "Как наследуется квартира?"
results = tagger.find_articles_by_new_sentence(query, k=10)

for rank, res in enumerate(results, 1):
    # Иконка в зависимости от score
    if res['score'] >= 0.6:
        icon = "🟢🔝"
    elif res['score'] >= 0.5:
        icon = "🟡"
    elif res['score'] >= 0.4:
        icon = "🔸"
    else:
        icon = "⚪"
    
    print(f"\n{icon} [{rank}] РЕЛЕВАНТНОСТЬ: {res['score']:.3f}")
    
    # Показываем начало статьи
    text_preview = res['text'][:200] + "..." if len(res['text']) > 200 else res['text']
    print(f"   {text_preview}")
    
    # Отмечаем статью 44, если она найдена
    if "наследование" in res['text'].lower():
        print("   ✅ ЭТО СТАТЬЯ 44!")
